# Avance 2. Ingeniería de caracteristicas

La fase de Ingeniería de Características (Feature Engineering) constituye un paso crítico dentro de la metodología CRISP-ML(Q), específicamente en la etapa de Data Engineering (Data Preparation). Según Studer et al. (2021), esta fase tiene como objetivo transformar los datos crudos en un conjunto de variables útiles para el aprendizaje automático, mitigando riesgos como el sesgo en las características y acelerando la convergencia de los algoritmos. 

En el contexto de nuestro proyecto, un sistema multi-agente basado en Retrieval-Augmented Generation (RAG) para auditorías ISO 27001, la ingeniería de características adquiere una dimensión particular. A diferencia de los proyectos de ML tabulares tradicionales, nuestro pipeline opera sobre datos textuales no estructurados que conforman la base de conocimiento del sistema. Por lo tanto, las técnicas de feature engineering se adaptan al dominio del procesamiento de lenguaje natural (NLP) y a los requerimientos específicos de los sistemas RAG.

## 1. Estado Actual de la Base de Conocimiento

Antes de definir el pipeline de ingeniería de características, es importante describir el estado actual de los datos disponibles. La base de conocimiento del proyecto está organizada en seis subcarpetas, cada una con un rol específico dentro del sistema multi-agente:

| Carpeta | Archivos | Tipo | Rol en el sistema |
|---|---|---|---|
| `knowledge-base/ciso-assistant-community/` | 4 YAML | Estructurado | Carga directa — definiciones de controles ISO en formato legible por máquina (CISO Assistant Library v2). Incluye los 93 controles de ISO 27001:2022, ISO 27001:2013, ISO 22301:2019 e ISO 42001:2023 con campo `ref_id` indexable. |
| `knowledge-base/iso27001/` | 1 PDF | No estructurado | Texto normativo completo de ISO/IEC 27001:2013 — fuente de verdad para citación en el pipeline RAG. |
| `knowledge-base/audit-examples/` | 7 PDF/DOCX | No estructurado | Reportes de auditoría reales para entrenar al agente en estructura y lenguaje de hallazgos. |
| `knowledge-base/audit-methodology/` | 4 PDF (español) | No estructurado | Guías ISO 19011 y modelos de madurez — marco metodológico para el agente de planeación. |
| `knowledge-base/sponsor-documents/` | 8 PDF/DOCX (español) | No estructurado | Políticas reales del patrocinador — evidencia de entrada para análisis de brecha y mapeo de controles. |
| `knowledge-base/non-conformities/` | 1 Markdown | Estructurado | Guía de las 10 no conformidades más frecuentes en ISO 27001 — conocimiento de dominio para el agente de hallazgos. |

**Total: 25 recursos.**

---

### ¿Por qué los YAML no necesitan RAG y los PDF/DOCX sí?

Los archivos YAML de `ciso-assistant-community/` tienen **estructura explícita**: cada control está definido como un registro con campos fijos (`ref_id`, `name`, `description`, `annotation`). El agente puede recuperar cualquier control con una búsqueda directa por clave — por ejemplo, `controles["A.5.1"]` — sin necesidad de similitud semántica. Es equivalente a consultar una base de datos relacional con una clave primaria conocida.

Los PDF y DOCX, en cambio, son **prosa no estructurada**: el contenido relevante puede estar en cualquier página, párrafo u oración, sin campos ni índices. Para que el agente pueda encontrar pasajes útiles (ej. "¿cómo documentó este auditor una no conformidad de control de acceso?"), es necesario:
1. Extraer el texto plano del documento.
2. Dividirlo en fragmentos (*chunks*) de tamaño manejable.
3. Convertir cada fragmento en un vector numérico (*embedding*) que capture su significado.
4. Almacenar esos vectores en ChromaDB para poder hacer búsquedas por similitud semántica en tiempo de ejecución.

Sin este proceso, el agente tendría que leer el documento entero en cada consulta, lo que supera el límite de contexto del modelo y eleva innecesariamente el costo por token.

> **Excepción — documentos del patrocinador:** estos 8 archivos sí se leen completos en cada ejecución del agente (sin pasar por ChromaDB) porque son relativamente cortos y es crítico que el agente tenga visibilidad total de la política, sin riesgo de que un fragmento relevante quede fuera del contexto por efecto de la fragmentación.

## 2. Transformación de Documentos en Embeddings y Almacenamiento en ChromaDB

Los documentos PDF y DOCX no estructurados deben pasar por un pipeline de transformación antes de poder ser consultados por los agentes. Este proceso convierte texto en bruto en representaciones numéricas (embeddings) que permiten búsqueda semántica eficiente.

Los documentos que siguen este pipeline son los de: `audit-examples/`, `audit-methodology/` e `iso27001/`. Los de `sponsor-documents/` se excluyen porque se leen completos en cada ejecución del agente.

### Pipeline de transformación

1. **Extracción de texto** — Se lee cada archivo con `pdfplumber` (PDF) o `python-docx` (DOCX) y se obtiene el texto plano página por página.
2. **Fragmentación (Chunking)** — El texto se divide en bloques de ~500 caracteres con un solapamiento de ~80 caracteres entre fragmentos consecutivos, para que ninguna idea quede partida sin contexto.
3. **Generación de embeddings** — Cada fragmento se convierte en un vector numérico de 384 dimensiones usando el modelo `paraphrase-multilingual-MiniLM-L12-v2` (sentence-transformers), que corre completamente en local sin costo de API.
4. **Persistencia en ChromaDB** — Los vectores se almacenan en disco junto con el texto original y metadatos (nombre de archivo, carpeta de origen, número de chunk) en `src/rag/chroma_db/`.

### Parámetros clave del chunking

| Parámetro | Valor | Justificación |
|---|---|---|
| Tamaño del chunk | ~500 caracteres | Suficiente contexto semántico sin exceder la ventana del modelo de embeddings |
| Solapamiento | ~80 caracteres | Evita que ideas que cruzan el límite de un chunk pierdan contexto |
| Modelo de embeddings | `paraphrase-multilingual-MiniLM-L12-v2` | Versión multilingüe del modelo MiniLM — mismas 384 dimensiones, ~118 MB, soporta español y 50+ idiomas adicionales. Reemplaza directamente a `all-MiniLM-L6-v2` sin cambios en el código. |

### Este proceso se ejecuta una sola vez

El script `src/rag/ingest.py` construye el índice y lo persiste en `src/rag/chroma_db/`. En ejecuciones posteriores el agente carga el índice desde disco directamente. Solo es necesario volver a ejecutar el script si se agregan nuevos documentos a la base de conocimiento.

## 3. Demostración del Pipeline con Archivos de la Base de Conocimiento

En esta sección importamos las funciones del script `src/rag/ingest.py` y ejecutamos cada paso del pipeline sobre archivos reales de la base de conocimiento, para verificar que el procesamiento funciona correctamente antes de lanzar la ingesta completa.

Archivos de prueba seleccionados:
- **PDF**: `knowledge-base/audit-methodology/planeacion-de-auditoria-prep-v1.1-2015.pdf` (guía de planeación de auditoría en español)
- **DOCX**: `knowledge-base/audit-examples/06100-FO8-Audit-Report_27001-2022_v4.0.docx` (plantilla oficial de reporte de auditoría ISO 27001:2022)

In [1]:
# Agregamos src/ al path de Python para poder importar los módulos del sistema
# sin necesidad de instalar el paquete formalmente.
import sys
from pathlib import Path

sys.path.insert(0, '../src')

# Importamos cada función del script de ingesta
from rag.ingest import (
    extract_pdf,
    extract_docx,
    extract_text,
    chunk_text,
    ingest_folder,
    build_index,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
)

KB = Path('../knowledge-base')
PDF_FILE  = KB / 'audit-methodology/planeacion-de-auditoria-prep-v1.1-2015.pdf'
DOCX_FILE = KB / 'audit-examples/06100-FO8-Audit-Report_27001-2022_v4.0.docx'

KB, PDF_FILE, DOCX_FILE

(WindowsPath('../knowledge-base'),
 WindowsPath('../knowledge-base/audit-methodology/planeacion-de-auditoria-prep-v1.1-2015.pdf'),
 WindowsPath('../knowledge-base/audit-examples/06100-FO8-Audit-Report_27001-2022_v4.0.docx'))

In [2]:
# Agrega esto para poder incorporar cambios a los archivos
%load_ext autoreload
%autoreload 2

### `extract_pdf` — Extracción de texto desde PDF

`pdfplumber` abre el archivo y recorre cada página extrayendo el texto como string. Las páginas sin texto (imágenes escaneadas) se omiten. El resultado es texto plano continuo con saltos de línea entre páginas.

In [3]:
texto_pdf = extract_pdf(PDF_FILE)

print(f'Archivo  : {PDF_FILE.name}')
print('---')
print(texto_pdf[:600])

Archivo  : planeacion-de-auditoria-prep-v1.1-2015.pdf
---
Planificación de la Auditoria
Universidad de Guanajuato- Departamento de Matemáticas
Jalisco S/N, Col. Valenciana CP: 36240 Guanajuato, Gto,
México.
Auditoria al Sistema Informático Programa de Resultados Electorales Preliminares- PREP
Auditoria al Sistema Informático Programa de Resultados Electorales Pre-
liminares- PREP
T + 52 473 7320006 / Ext. 1203 F +52 7320006 W demat.ugto.mx
Diseño del Sistema Informático de Gestión Judicial Penal Federal
1
Tabla de Contenido
Tabla de Contenido 1
Lista de Cuadros 2
Lista de Figuras 2
Información General 3
Nombre del Proyecto 3
Roles de la Auditoría 4
P


### `extract_docx` — Extracción de texto desde DOCX

`python-docx` lee el documento párrafo por párrafo y descarta los que están vacíos. Los párrafos resultantes se unen con saltos de línea. Este enfoque es más limpio que extraer el XML directamente porque respeta la estructura lógica del documento.

In [4]:
texto_docx = extract_docx(DOCX_FILE)

print(f'Archivo  : {DOCX_FILE.name}')
print(f'\n--- Primeros 600 caracteres ---')
print(texto_docx[:600])

Archivo  : 06100-FO8-Audit-Report_27001-2022_v4.0.docx

--- Primeros 600 caracteres ---
DISTRIBUTION
The content of this report must not be disclosed to a third party without the agreement of the MSECB Client.
DISCLAIMER
This report has been prepared by MSECB in respect of a client’s application for assessment by MSECB. The purpose of the report is to verify the Client's conformance with the management system standard(s) or other criteria specified. The content of this report applies only to matters, which were evident to MSECB at the time of the audit within the audit scope. MSECB does not warrant or otherwise comment upon the suitability of the contents of the report or the cer


### `extract_text` — Dispatcher por extensión

Esta función actúa como punto de entrada único: recibe cualquier archivo soportado y delega a `extract_pdf` o `extract_docx` según su extensión. Si la extensión no es `.pdf` ni `.docx`, devuelve un string vacío en lugar de lanzar una excepción, lo que permite continuar la ingesta aunque haya archivos inesperados en la carpeta.

In [5]:
# extract_text delega internamente a extract_pdf o extract_docx
for archivo in [PDF_FILE, DOCX_FILE]:
    texto = extract_text(archivo)
    print(f'{archivo.suffix.upper()}  {archivo.name}')

.PDF  planeacion-de-auditoria-prep-v1.1-2015.pdf
.DOCX  06100-FO8-Audit-Report_27001-2022_v4.0.docx


### `chunk_text` — Fragmentación con solapamiento

El texto completo se divide en bloques de `CHUNK_SIZE` caracteres. Cada bloque avanza `CHUNK_SIZE - CHUNK_OVERLAP` caracteres respecto al anterior, por lo que los últimos `CHUNK_OVERLAP` caracteres de un chunk coinciden con el inicio del siguiente. Esto evita que una frase que cruza el límite de un bloque quede semánticamente incompleta en ambos extremos.

In [6]:
chunks = chunk_text(texto_pdf)
chunks[0], len(chunks)

('Planificación de la Auditoria\nUniversidad de Guanajuato- Departamento de Matemáticas\nJalisco S/N, Col. Valenciana CP: 36240 Guanajuato, Gto,\nMéxico.\nAuditoria al Sistema Informático Programa de Resultados Electorales Preliminares- PREP\nAuditoria al Sistema Informático Programa de Resultados Electorales Pre-\nliminares- PREP\nT + 52 473 7320006 / Ext. 1203 F +52 7320006 W demat.ugto.mx\nDiseño del Sistema Informático de Gestión Judicial Penal Federal\n1\nTabla de Contenido\nTabla de Contenido 1\nLista d',
 89)

### `ingest_folder` — Ingesta de una carpeta completa

Itera sobre todos los PDF/DOCX de la carpeta, extrae y fragmenta cada archivo, y llama a `collection.upsert()` para almacenar los chunks junto con sus vectores y metadatos en ChromaDB. El uso de `upsert` (en lugar de `add`) hace que la función sea idempotente: volver a ejecutarla sobre los mismos archivos actualiza los registros existentes sin duplicarlos.

Antes de llamar a `ingest_folder`, la celda siguiente inicializa tres objetos necesarios:

- **`client`** — Conexión al motor de ChromaDB. `PersistentClient` indica que la base de datos se guarda en disco (en `src/rag/chroma_db/`) y sobrevive entre ejecuciones. Es el equivalente a abrir una conexión a una base de datos relacional.

- **`ef` (embedding function)** — Función que convierte texto en vectores numéricos. Se le pasa el modelo `paraphrase-multilingual-MiniLM-L12-v2` para que ChromaDB sepa cómo transformar cada chunk antes de almacenarlo. Sin este parámetro, ChromaDB no sabría representar el texto en el espacio vectorial.

- **`collection`** — El espacio de almacenamiento dentro de ChromaDB donde se guardan los chunks. Una **colección** es análoga a una tabla en una base de datos relacional: agrupa documentos del mismo dominio (en este caso, todos los fragmentos de la base de conocimiento ISO 27001) y permite hacer búsquedas semánticas sobre ellos. `get_or_create_collection` devuelve la colección si ya existe o la crea nueva si no, lo que hace la operación segura de ejecutar varias veces.

> Esta función escribe en disco (`src/rag/chroma_db/`). La celda siguiente es demostrativa — para construir el índice completo ejecuta `python src/rag/ingest.py` desde la raíz del proyecto.

In [7]:
%%time
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rag.ingest import EMBEDDING_MODEL, COLLECTION_NAME, CHROMA_PATH

# Inicializamos ChromaDB apuntando a src/rag/chroma_db/
client = chromadb.PersistentClient(path=str(CHROMA_PATH))
ef = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
collection = client.get_or_create_collection(COLLECTION_NAME, embedding_function=ef)

# Procesamos solo la carpeta audit-methodology como prueba
folder = KB / 'audit-methodology'
chunks_insertados = ingest_folder(collection, folder, source_label='audit_methodology')

print(f'\nChunks insertados en esta ejecución : {chunks_insertados}')
print(f'Total documentos en la colección    : {collection.count()}')

C:\Users\spoc3\Documents\data-science\proyecto-integrador\valkiria_auditor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 199/199 [00:00<00:00, 6


2.1_requisitos_norma_iso_19011-2011.pdf → 317 chunks
ieeg-resultados-de-auditoria-en-madurez-ti-v1.2-2015.pdf → 1050 chunks
planeacion-de-auditoria-prep-v1.1-2015.pdf → 89 chunks
plexus-diagnostico_pss_iaft16949-2018.pdf → 390 chunks

Chunks insertados en esta ejecución : 1846
Total documentos en la colección    : 1846
CPU times: total: 5min 49s
Wall time: 1min 43s


### `search_collection` — Consulta semántica a ChromaDB

Una vez construido el índice, podemos consultarlo con lenguaje natural. `search_collection` convierte la pregunta en un vector usando el mismo modelo de embeddings que se usó durante la ingesta, busca los vectores más cercanos en ChromaDB, y devuelve los chunks más similares rankeados por su **score de similitud coseno** (0 a 1, donde 1 es idéntico).

**¿Cómo funciona la similitud coseno?**
Dos textos semánticamente relacionados producen vectores que apuntan en direcciones similares en el espacio de 384 dimensiones. La similitud coseno mide el ángulo entre ellos: un ángulo de 0° (misma dirección) da score 1.0, un ángulo de 90° (sin relación) da score 0.0. Esto permite encontrar fragmentos relevantes aunque no compartan palabras exactas con la consulta.

In [8]:
from rag.ingest import search_collection, print_results

# Consulta en español — el modelo multilingüe maneja esto sin configuración adicional
query = 'procedimiento para documentar hallazgos y no conformidades en una auditoría'

results = search_collection(query, n_results=4)
print_results(results)

[#1] score=0.8093 | planeacion-de-auditoria-prep-v1.1-2015.pdf (chunk 59)
     folder : audit_methodology
     excerpt: s .
!Auditoría. !Proceso sistemático, independiente y documentado para obtener evidencias de la auditoría y
evaluarlas de manera objetiva con el fin de determinar la extensión en que se cumplen los criterios de auditoría.
Criterios de auditoría!. Grupo de políticas, procedimientos o requisitos usado...

[#2] score=0.8064 | 2.1_requisitos_norma_iso_19011-2011.pdf (chunk 176)
     folder : audit_methodology
     excerpt: es menos formal y puede consistir
solo en comunicar los hallazgos y conclusiones de la auditoría.
Según sea apropiado, se debe explicar lo siguiente al auditado durante la reunión de
cierre:
— prevenir respecto a que la evidencia de auditoría recolectada está basada en una
muestra de la información...

[#3] score=0.7996 | 2.1_requisitos_norma_iso_19011-2011.pdf (chunk 166)
     folder : audit_methodology
     excerpt: lo B.6. provee lineamientos sobre l

#### Interpretación de los resultados

Cada resultado muestra:
- **score** — qué tan semánticamente cercano es el chunk a la consulta (más alto = más relevante)
- **source** — el archivo de origen del fragmento
- **folder** — de qué subcarpeta de la base de conocimiento proviene
- **chunk** — índice del fragmento dentro del documento (útil para localizar el contexto original)
- **excerpt** — los primeros 300 caracteres del fragmento recuperado

Un score ≥ 0.75 indica alta relevancia semántica. Scores < 0.5 sugieren que el índice no contiene información directamente relacionada con la consulta.

### Cómo se usa esto en el pipeline RAG

La búsqueda semántica es el puente entre la base de conocimiento y el agente. El flujo completo funciona así:

1. **El agente recibe una tarea**. Por ejemplo, evaluar si la política de control de acceso del patrocinador cumple con el control A.8.3 de ISO 27001.

2. **Consulta ChromaDB**. Llama a `search_collection` con una pregunta relacionada (*'ejemplos de hallazgos de control de acceso en auditorías ISO 27001'*) para recuperar fragmentos relevantes de los reportes de auditoría reales.

3. **Construye el contexto**. Los chunks recuperados se concatenan y se incluyen en el prompt del modelo junto con el texto de la política del patrocinador.

4. **El modelo genera la respuesta**. Con acceso tanto a la política real como a ejemplos de cómo auditores documentaron hallazgos similares, el modelo produce un hallazgo fundamentado y con lenguaje de auditoría correcto.


### `build_index` — Orquestación completa

`build_index` combina todas las funciones anteriores en una sola llamada: crea (o reutiliza) la colección en ChromaDB y llama a `ingest_folder` para cada una de las tres carpetas de referencia (`audit-examples/`, `audit-methodology/`, `iso27001/`). El argumento `reset=True` elimina la colección existente antes de reconstruirla, lo que resulta útil cuando se actualizan documentos de la base de conocimiento.

```bash
# Ingesta completa desde la raíz del proyecto
python src/rag/ingest.py

# Reconstrucción desde cero
python src/rag/ingest.py --reset
```